# Study signal effs...

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime

import uproot
from matplotlib import gridspec

import sys
sys.path.append('../../../../')
from pyanalib.split_df_helpers import *
import pyanalib.pandas_helpers as ph
import pyanalib.stat_helpers as sh
from makedf.util import *
from analysis_village.plot_style.plot_helper import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
sample_str = "v10_14_02_02+"

## Open MC file

In [ ]:
input_path = "/data/sungbino/sbnd/gen2/cohpi/"
mc_file_path = input_path + "aurora_SBND2026A_gen2_BNBLight_prodgenie_corsika_proton_rockbox0p1_sbnd_CV_v10_14_02_03_flatcaf_sbnd_pandora_opflash_100files.df"
print("mc n split", get_n_split(mc_file_path))
print_keys(mc_file_path)


In [ ]:
keys2load = ['hdr', "mcnu", 'evt', "pot"]
mc_bnb_cosmic_dfs = load_dfs(mc_file_path, keys2load, n_max_concat=4)


In [ ]:
mc_tot_pot = mc_bnb_cosmic_dfs["hdr"]['pot'].sum()
print("mc tot pot", mc_tot_pot)

In [ ]:
mc_bnb_cosmic_dfs['evt']

In [ ]:
mc_bnb_cosmic_dfs['mcnu']

## Select true mcnu within FV and CCQE

In [ ]:
def InFV_nohiyz(data):
    xmin = 10.
    xmax = 190.
    zmin = 10.
    zmax = 450.
    ymax_highz = 100.
    pass_xz = (np.abs(data.x) > xmin) & (np.abs(data.x) < xmax) & (data.z > zmin) & (data.z < zmax)
    pass_y = ((data.z < 250) & (np.abs(data.y) < 190.)) | ((data.z > 250) & (data.y > -190.) & (data.y < ymax_highz))
    return pass_xz & pass_y

In [ ]:
true_in_fv = InFV_nohiyz(mc_bnb_cosmic_dfs['mcnu'].position)
mc_bnb_cosmic_dfs['mcnu'] = mc_bnb_cosmic_dfs['mcnu'][true_in_fv]

In [ ]:
#mc_bnb_cosmic_dfs['mcnu'] = mc_bnb_cosmic_dfs['mcnu'][(mc_bnb_cosmic_dfs['mcnu'].genie_mode == 0) & (mc_bnb_cosmic_dfs['mcnu'].iscc)]

In [ ]:
mc_bnb_cosmic_dfs['mcnu']

## Check neutrino cos theta

In [ ]:
mc_bnb_cosmic_dfs['mcnu'][('dir', 'x', '')] = mc_bnb_cosmic_dfs['mcnu'].momentum.x / (np.sqrt(mc_bnb_cosmic_dfs['mcnu'].momentum.x**2 + mc_bnb_cosmic_dfs['mcnu'].momentum.y**2 + mc_bnb_cosmic_dfs['mcnu'].momentum.z**2))
mc_bnb_cosmic_dfs['mcnu'][('dir', 'y', '')] = mc_bnb_cosmic_dfs['mcnu'].momentum.y / (np.sqrt(mc_bnb_cosmic_dfs['mcnu'].momentum.x**2 + mc_bnb_cosmic_dfs['mcnu'].momentum.y**2 + mc_bnb_cosmic_dfs['mcnu'].momentum.z**2))
mc_bnb_cosmic_dfs['mcnu'][('dir', 'z', '')] = mc_bnb_cosmic_dfs['mcnu'].momentum.z / (np.sqrt(mc_bnb_cosmic_dfs['mcnu'].momentum.x**2 + mc_bnb_cosmic_dfs['mcnu'].momentum.y**2 + mc_bnb_cosmic_dfs['mcnu'].momentum.z**2))

In [ ]:
mc_bnb_cosmic_dfs['mcnu'][('theta', '', '')] = np.arccos(mc_bnb_cosmic_dfs['mcnu'][('dir', 'z', '')]) * 180 / np.pi

In [ ]:
draw_a_distribution(mc_bnb_cosmic_dfs['mcnu'][mc_bnb_cosmic_dfs['mcnu'].pdg==12], ('theta', '', ''), x_min=0., x_max=4,  nbins=40, title_x=r"$\theta_{\nu}^{true}$ (degrees)", title_y="Interacting Neutrinos in FV", label_top=r"$\nu_e$", out_name="theta_nu_true_nue.pdf")
draw_a_distribution(mc_bnb_cosmic_dfs['mcnu'][mc_bnb_cosmic_dfs['mcnu'].pdg==-12], ('theta', '', ''), x_min=0., x_max=4,  nbins=40, title_x=r"$\theta_{\nu}^{true}$ (degrees)", title_y="Interacting Neutrinos in FV", label_top=r"$\bar{\nu}_e$", out_name="theta_nu_true_nuebar.pdf")
draw_a_distribution(mc_bnb_cosmic_dfs['mcnu'][mc_bnb_cosmic_dfs['mcnu'].pdg==14], ('theta', '', ''), x_min=0., x_max=4,  nbins=40, title_x=r"$\theta_{\nu}^{true}$ (degrees)", title_y="Interacting Neutrinos in FV", label_top=r"$\nu_\mu$", out_name="theta_nu_true_numu.pdf")
draw_a_distribution(mc_bnb_cosmic_dfs['mcnu'][mc_bnb_cosmic_dfs['mcnu'].pdg==-14], ('theta', '', ''), x_min=0., x_max=4,  nbins=40, title_x=r"$\theta_{\nu}^{true}$ (degrees)", title_y="Interacting Neutrinos in FV", label_top=r"$\bar{\nu}_\mu$", out_name="theta_nu_true_numubar.pdf")

## compare true nu dir and dir based on the target position

In [ ]:
mc_bnb_cosmic_dfs['mcnu'].position

In [ ]:
mc_bnb_cosmic_dfs['mcnu'][('vec', 'fromtarget', 'x')] = mc_bnb_cosmic_dfs['mcnu'].position.x + 73.78
mc_bnb_cosmic_dfs['mcnu'][('vec', 'fromtarget', 'y')] = mc_bnb_cosmic_dfs['mcnu'].position.y
mc_bnb_cosmic_dfs['mcnu'][('vec', 'fromtarget', 'z')] = mc_bnb_cosmic_dfs['mcnu'].position.z + 11000.

In [ ]:
mc_bnb_cosmic_dfs['mcnu'][('dir', 'fromtarget', 'x')] = mc_bnb_cosmic_dfs['mcnu'][('vec', 'fromtarget', 'x')] / (np.sqrt(mc_bnb_cosmic_dfs['mcnu'][('vec', 'fromtarget', 'x')]**2 + mc_bnb_cosmic_dfs['mcnu'][('vec', 'fromtarget', 'y')]**2 + mc_bnb_cosmic_dfs['mcnu'][('vec', 'fromtarget', 'z')]**2))
mc_bnb_cosmic_dfs['mcnu'][('dir', 'fromtarget', 'y')] = mc_bnb_cosmic_dfs['mcnu'][('vec', 'fromtarget', 'y')] / (np.sqrt(mc_bnb_cosmic_dfs['mcnu'][('vec', 'fromtarget', 'x')]**2 + mc_bnb_cosmic_dfs['mcnu'][('vec', 'fromtarget', 'y')]**2 + mc_bnb_cosmic_dfs['mcnu'][('vec', 'fromtarget', 'z')]**2))
mc_bnb_cosmic_dfs['mcnu'][('dir', 'fromtarget', 'z')] = mc_bnb_cosmic_dfs['mcnu'][('vec', 'fromtarget', 'z')] / (np.sqrt(mc_bnb_cosmic_dfs['mcnu'][('vec', 'fromtarget', 'x')]**2 + mc_bnb_cosmic_dfs['mcnu'][('vec', 'fromtarget', 'y')]**2 + mc_bnb_cosmic_dfs['mcnu'][('vec', 'fromtarget', 'z')]**2))

In [ ]:
mc_bnb_cosmic_dfs['mcnu']['costheta', 'true_between_fromtarget', ''] = mc_bnb_cosmic_dfs['mcnu'][('dir', 'z', '')] * mc_bnb_cosmic_dfs['mcnu'][('dir', 'fromtarget', 'z')] + mc_bnb_cosmic_dfs['mcnu'][('dir', 'x', '')] * mc_bnb_cosmic_dfs['mcnu'][('dir', 'fromtarget', 'x')] + mc_bnb_cosmic_dfs['mcnu'][('dir', 'y', '')] * mc_bnb_cosmic_dfs['mcnu'][('dir', 'fromtarget', 'y')]

In [ ]:
mc_bnb_cosmic_dfs['mcnu']['theta', 'true_between_fromtarget', ''] = np.arccos(mc_bnb_cosmic_dfs['mcnu']['costheta', 'true_between_fromtarget', '']) * 180 / np.pi

In [ ]:
draw_a_distribution(mc_bnb_cosmic_dfs['mcnu'][mc_bnb_cosmic_dfs['mcnu'].pdg==12], ('theta', 'true_between_fromtarget', ''), x_min=0., x_max=4,  nbins=40, title_x=r"$\theta({\nu}^{true} - \text{from target})$ (degrees)", title_y="Interacting Neutrinos in FV", label_top=r"$\nu_e$", out_name="theta_diff_nu_true_nue.pdf")
draw_a_distribution(mc_bnb_cosmic_dfs['mcnu'][mc_bnb_cosmic_dfs['mcnu'].pdg==14], ('theta', 'true_between_fromtarget', ''), x_min=0., x_max=4,  nbins=40, title_x=r"$\theta({\nu}^{true} - \text{from target})$ (degrees)", title_y="Interacting Neutrinos in FV", label_top=r"$\nu_\mu$", out_name="theta_diff_nu_true_numu.pdf")


## Collect evt dfs that matches to ccqe events

In [ ]:
mc_bnb_cosmic_dfs["mcnu"].columns = pd.MultiIndex.from_tuples([('gen',) + col if isinstance(col, tuple) else ('gen', col) for col in mc_bnb_cosmic_dfs["mcnu"].columns])
mc_bnb_cosmic_dfs["mcnu"].columns = pd.MultiIndex.from_tuples([
    col + ('',) * (6 - len(col)) for col in mc_bnb_cosmic_dfs["mcnu"].columns
])
mc_bnb_cosmic_dfs["evt"] = ph.multicol_merge(mc_bnb_cosmic_dfs["evt"].reset_index(), mc_bnb_cosmic_dfs["mcnu"].reset_index(),
                            left_on=[('__ntuple', '', '', '', '', ''), ('entry', '', '', '', '', ''), ('slc','tmatch', 'idx', '', '', '')],
                            right_on=[('__ntuple', '', '', '', '', ''), ('entry', '', '', '', '', ''), ('rec.mc.nu..index', '','', '', '', '')], 
                            how="left") ## -- save all 
mc_bnb_cosmic_dfs["evt"] = mc_bnb_cosmic_dfs["evt"].set_index(["__ntuple", "entry", "rec.slc..index", "rec.slc.reco.pfp..index"], verify_integrity=True)

In [ ]:
mc_bnb_cosmic_dfs["evt"] = mc_bnb_cosmic_dfs["evt"].dropna(subset=[('gen', 'E', '', '', '', '')]) ## -- drop events that don't have matched mcnu

In [ ]:
mc_bnb_cosmic_dfs['evt'][('pfp', 'trk', 'truth', 'p', 'genp', 'p')] = np.sqrt(mc_bnb_cosmic_dfs['evt'][('pfp', 'trk', 'truth', 'p', 'genp', 'x')]**2 + mc_bnb_cosmic_dfs['evt'][('pfp', 'trk', 'truth', 'p', 'genp', 'y')]**2 + mc_bnb_cosmic_dfs['evt'][('pfp', 'trk', 'truth', 'p', 'genp', 'z')]**2)
mc_bnb_cosmic_dfs['mcnu'][('gen', 'mu', 'genp', 'p', '', '')] = np.sqrt(mc_bnb_cosmic_dfs['mcnu'][('gen', 'mu', 'genp', 'x', '', '')]**2 + mc_bnb_cosmic_dfs['mcnu'][('gen', 'mu', 'genp', 'y', '', '')]**2 + mc_bnb_cosmic_dfs['mcnu'][('gen', 'mu', 'genp', 'z', '', '')]**2)

### collect only reco muon pfp

In [ ]:
mc_bnb_cosmic_dfs["evt"].pfp.trk.truth.p.pdg

In [ ]:
mu_evt = mc_bnb_cosmic_dfs["evt"][np.abs(mc_bnb_cosmic_dfs["evt"].pfp.trk.truth.p.pdg) == 13]

In [ ]:
mu_evt

Select best tmatch slice

In [ ]:
tmatch_eff_col = mu_evt[("slc", "tmatch", "eff", "", "", "")]
mu_evt = mu_evt.groupby([
    mu_evt.index.get_level_values('__ntuple'),
    mu_evt.index.get_level_values('entry'),
    mu_evt.index.get_level_values('rec.slc..index'),
    tmatch_eff_col
]).head(1)

Select only one pfp when multiple pfps are matched to single true muon

In [ ]:
mu_evt_genp_col = mu_evt.pfp.trk[('truth', 'p', 'genp', 'p')]
mu_evt = mu_evt.pfp.trk.groupby([
    mu_evt.pfp.trk.index.get_level_values('__ntuple'),
    mu_evt.pfp.trk.index.get_level_values('entry'),
    mu_evt.pfp.trk.index.get_level_values('rec.slc..index'),
    mu_evt_genp_col
]).head(1)

In [ ]:
mu_evt

## Check mcnu position xyz

In [ ]:
draw_a_distribution(mc_bnb_cosmic_dfs['evt'], ("slc", "vertex", "x", "", "", ""), "True Vertex X [cm]", -210., 210., 22, "Events")
draw_a_distribution(mc_bnb_cosmic_dfs['evt'], ("slc", "vertex", "y", "", "", ""), "True Vertex Y [cm]", -210., 210., 22, "Events")
draw_a_distribution(mc_bnb_cosmic_dfs['evt'], ("slc", "vertex", "z", "", "", ""), "True Vertex Z [cm]", -10., 510., 52, "Events")

## Collect efficiency

In [ ]:
gen_ke_min = 0
gen_ke_max = 0.1
n_gen_ke_bins = 20

In [ ]:
mu_evt.truth.p.columns

In [ ]:
true_mu_genp = mc_bnb_cosmic_dfs['mcnu'][('gen', 'mu', 'genp', 'p', '', '')]
evt_mu_genp = mu_evt[('truth', 'p', 'genp', 'p')]

In [ ]:
true_mu_gen_ke = np.sqrt(true_mu_genp**2 + 0.1056583755**2) - 0.1056583755
evt_mu_gen_ke = np.sqrt(evt_mu_genp**2 + 0.1056583755**2) - 0.1056583755

In [ ]:
true_mu_gen_ke_hist = np.histogram(true_mu_gen_ke, bins=n_gen_ke_bins, range=(gen_ke_min, gen_ke_max))
evt_mu_gen_ke_hist = np.histogram(evt_mu_gen_ke, bins=n_gen_ke_bins, range=(gen_ke_min, gen_ke_max))

In [ ]:
true_mu_gen_ke_hist

In [ ]:
evt_mu_gen_ke_hist

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def plot_efficiencies_from_hists(
    hist_dict,
    xlabel="Kinetic Energy (GeV)",
    ylabel="Efficiency (Reconstructed as PFP / Simulated)",
    title="Muon PFP Efficiency",
    figsize=(8, 6),
    out_name=None,
):
    """Calculates efficiencies with binomial errors from np.histogram outputs and plots them directly.

    Parameters:
        hist_dict (dict): Maps labels to configurations -> {label: (evt_hist, true_hist, color, marker)}
        xlabel (str): Label for the x-axis.
        ylabel (str): Label for the y-axis.
        title (str): Title of the plot.
        figsize (tuple): Dimensions of the figure.
    """
    plt.figure(figsize=figsize)

    for label, config in hist_dict.items():
        # Unpack inputs for this specific particle type
        evt_hist, true_hist = config[0], config[1]
        color = config[2] if len(config) > 2 else None
        marker = config[3] if len(config) > 3 else "o"

        evt_counts, _ = evt_hist
        true_counts, bin_edges = true_hist

        # Calculate bin geometry
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
        bin_widths = 0.5 * (bin_edges[1:] - bin_edges[:-1])

        # Convert to float for precise division
        evt_counts = evt_counts.astype(float)
        true_counts = true_counts.astype(float)

        # Safely calculate efficiency (defaults to 0 where true counts are 0)
        eff = np.divide(
            evt_counts,
            true_counts,
            out=np.zeros_like(true_counts),
            where=true_counts > 0,
        )

        # Safely calculate standard binomial uncertainties
        variance = np.divide(
            eff * (1.0 - eff),
            true_counts,
            out=np.zeros_like(eff),
            where=true_counts > 0,
        )
        err = np.sqrt(variance)

        # Add to the plot
        plt.errorbar(
            bin_centers,
            eff,
            yerr=err,
            xerr=bin_widths,
            fmt=marker,
            label=f"{label} Efficiency",
            capsize=3,
            color=color,
        )

    # Finalize plot formatting
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.ylim(0.0, 1.05)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend(loc="lower right")

    plt.tight_layout()
    if out_name:
        plt.savefig(out_name)
    plt.show()

In [ ]:
my_histograms = {
    "Muon": (evt_mu_gen_ke_hist, true_mu_gen_ke_hist, "blue", "o"),
}
plot_efficiencies_from_hists(
    my_histograms,
    xlabel="Kinetic Energy (GeV)",  # Adjust units as needed
    title="Muons from CCQE",
    out_name="muon_ccqe_efficiency.pdf"
)